In [5]:
# Load Data & Clean Missing Values
import pandas as pd
import numpy as np

# 1. Load dataset
df = pd.read_csv('/Users/gayatrishankar/Downloads/emi_prediction_dataset.csv', low_memory=False)

# 2. Define categorical columns
categorical_cols = [
    'gender', 'marital_status', 'education', 
    'employment_type', 'company_type', 'house_type', 
    'emi_scenario', 'emi_eligibility'
]

# 3. Convert all other numerical columns safely
for col in df.columns:
    if col not in categorical_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# 4. Fill missing numerical values (median first, fallback to 0)
num_cols = df.select_dtypes(include=[np.number]).columns
for col in num_cols:
    median_val = df[col].median()
    if pd.isna(median_val):
        df[col] = df[col].fillna(0)
    else:
        df[col] = df[col].fillna(median_val)

# 5. Fill missing categorical values with mode
cat_cols = df.select_dtypes(include=['object']).columns
for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

print("✅ Data cleaning complete!")
print(f"Remaining Missing Values: {df.isnull().sum().sum()}")

✅ Data cleaning complete!
Remaining Missing Values: 0


In [6]:
# Feature Engineering (Financial Ratios)
# Create new financial features
df['total_monthly_expenses'] = (
    df['monthly_rent'] + 
    df['school_fees'] + 
    df['college_fees'] + 
    df['travel_expenses'] + 
    df['groceries_utilities'] + 
    df['other_monthly_expenses'] + 
    df['current_emi_amount']
)

df['disposable_income'] = df['monthly_salary'] - df['total_monthly_expenses']
df['dti_ratio'] = np.where(df['monthly_salary'] > 0, df['current_emi_amount'] / df['monthly_salary'], 0)
df['expense_to_income_ratio'] = np.where(df['monthly_salary'] > 0, df['total_monthly_expenses'] / df['monthly_salary'], 0)

print("✅ Feature engineering completed!")
print(f"New Dataset Shape: {df.shape}")

✅ Feature engineering completed!
New Dataset Shape: (404800, 31)


In [7]:
# Categorical Encoding & Train-Test Split
from sklearn.preprocessing import LabelEncoder

# Encode categorical variables using LabelEncoder
le_dict = {}
cat_cols = ['gender', 'marital_status', 'education', 'employment_type', 'company_type', 'house_type', 'emi_scenario', 'emi_eligibility']

for col in cat_cols:
    if col in df.columns:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))
        le_dict[col] = le

# Save cleaned and processed dataset for model training
df.to_csv('data/processed_emi_dataset.csv', index=False)
print("✅ Categorical encoding complete! Processed dataset saved to 'data/processed_emi_dataset.csv'")

✅ Categorical encoding complete! Processed dataset saved to 'data/processed_emi_dataset.csv'
